# Investigate the decoded spots using TissUUmaps SpotInspector plug-in.  

**SpotInspector export**


This notebook takes:
1) a **codebook CSV** (its **first column** is treated as the `target` key), and  
2) a **decoded `fov.csv`** (must contain a `target` column).

It outputs a new CSV with three added columns:
- `code`: semicolon-separated codes across cycles (e.g. `1;2;1;4`)
- `cycles`: the same for all rows (e.g. `0;1;2;3`)
- `spotinspector`: same as `code` but zero-indexed (e.g. `0;1;0;3`)

This enables you to open all the SpaceTX formatted images from that FOV (primary, not nuclei!) in Tissumaps and use SpotInspector plug-in to investigate your decoded spots.

Contact person: Augusta Jensen

**Please ask me if you have questions on how to use it in TissUUmaps /Augusta**

In [ ]:
import os
import pandas as pd

# ========= INPUTS =========
codebook_path = 'path/to/codebook.csv' 
fov_path      = 'path/to/R1/decoding/2_decoded/fov_004.csv'      
n_cycles      = 5  
# ==========================

In [11]:
# Load inputs
codebook = pd.read_csv(codebook_path, header=None)
fov = pd.read_csv(fov_path)

# Basic checks
if "target" not in fov.columns:
    raise ValueError("fov.csv must contain a column named 'target'.")

# The codebook's FIRST column is the key that maps to fov['target']
key_col = 0

# The rest of the columns are cycles, in order: 1..n_cycles
cycle_cols = list(range(1, n_cycles + 1))

# Validate codebook has enough columns
needed_max_col = n_cycles  # because key_col=0 + cycles are 1..n_cycles
if codebook.shape[1] <= needed_max_col:
    raise ValueError(
        f"Codebook has {codebook.shape[1]} columns, but you requested n_cycles={n_cycles}. "
        f"Expected at least {needed_max_col + 1} columns (0..{needed_max_col})."
    )

# Normalize types
codebook[key_col] = codebook[key_col].astype(str).str.strip()
for c in cycle_cols:
    # Keep integers clean; allow missing as <NA>
    codebook[c] = pd.to_numeric(codebook[c], errors="coerce").astype("Int64")

fov["target"] = fov["target"].astype(str).str.strip()

In [ ]:
# Keep only target + cycles
codebook_out = codebook[[key_col] + cycle_cols].copy()

# code: "c1;c2;...;cN"
codebook_out["code"] = codebook_out[cycle_cols].astype(str).agg(";".join, axis=1)

# cycles: same for all rows: "0;1;...;n-1"
cycles_str = ";".join(map(str, range(n_cycles)))
codebook_out["cycles"] = cycles_str

# spotinspector: (cycle values - 1), then join the same way
codebook_out["spotinspector"] = (codebook_out[cycle_cols] - 1).astype(str).agg(";".join, axis=1)

codebook_out.head()


In [ ]:
# Merge onto the fov table
merged = fov.merge(
    codebook_out[[key_col, "code", "cycles", "spotinspector"]],
    left_on="target",
    right_on=key_col,
    how="left",
)

# Optional: drop the duplicated key column from codebook
if key_col != "target":
    merged = merged.drop(columns=[key_col])

merged.head()


In [ ]:
# Save next to the input fov file
out_path = os.path.join(
    os.path.dirname(os.path.abspath(fov_path)),
    os.path.splitext(os.path.basename(fov_path))[0] + "_SpotInspector.csv",
)
merged.to_csv(out_path, index=False)
out_path